In [ ]:
%%configure -f
{"vCores": 4, "defaultLakehouse": {"name": "diagnostic", "id": "9d10bce5-1edc-4875-83c4-ac0a98a02775", "workspaceId": "82ad2591-974a-4ad4-ace6-e24879274a4b"}}


# Reflection v2 A/B — Arm B (treatment: medium effort, v2 reflection prompt ON)

**Branch:** `experiment/reflection-v2`  
**Run ID (shared across all 4 arms):** `reflection-ab-3arm-20260504-095812`  
**Arm label:** `B_medium_v2`

## Configuration
- **Effort tier:** medium
- **Reflection:** ON
- **Reflection prompt:** v2 (current `fabric_rlm.prompts.build_reflection_prompt`: default-approve, two narrow checks, minimal-edit)
- **Dataset:** `longcot_cs_hard_holdout25.jsonl` (n=25)
- **Wheel:** `fabric_rlm-0.1.11.dev13+reflectionv2-py3-none-any.whl`
- **Engine:** plain `RLM` (no bandit / no decompose) so reflection is the only varying knob

## Purpose
Primary treatment arm. Tests whether v2's default-approve + targeted-checks design preserves wins while suppressing the harmful over-revisions seen in dev11.

## Output location
`abfss://sandeep_ws@onelake.dfs.fabric.microsoft.com/diagnostic.Lakehouse/Files/fabric_rlm_adaptive_validation/reflection_ab/reflection-ab-3arm-20260504-095812/B_medium_v2/`  
└─ `results.jsonl`, `arm_summary.json`, `traces/trace_*.json`

## Sibling arms (parallel runs, same RUN_ID)
- `A_medium_v1` (effort=medium, reflection=ON)
- `C_medium_off` (effort=medium, reflection=OFF (`enable_reflection=False`))
- `B_high_v2_sanity` (effort=high, reflection=ON)

After all 4 arms complete, run the decision-rule cell from `reflection_v2_AB_3arm.ipynb` (or the local equivalent) against the shared RUN_ID directory to apply the SHIP_V2 / FLIP_OFF rule from `RESEARCH-reflection-v2-ab.md`.


In [ ]:
import os, sys, json, time, traceback, uuid, platform as _plat, subprocess, re
from pathlib import Path
WHEEL_PATH = "/lakehouse/default/Files/fabric_rlm_longcot/wheels/fabric_rlm-0.1.11.dev13+reflectionv2-py3-none-any.whl"
DATASET_PATH = "/lakehouse/default/Files/fabric_rlm_longcot/datasets/longcot_cs_hard_holdout25.jsonl"
RUN_ID = "reflection-ab-3arm-20260504-095812"
TIER = "reflection_ab"
FILES_ROOT = Path("/lakehouse/default/Files")
RUN_ROOT = FILES_ROOT / "fabric_rlm_adaptive_validation" / TIER / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)
SMOKE_N = None

def stage(name, **info):
    line = {"stage": name, **info, "t": round(time.time()-_t0, 1)}
    print("[stage]", json.dumps(line, default=str))
_t0 = time.time()
stage("setup", run_root=str(RUN_ROOT))
subprocess.call(["pip","uninstall","-y","-q","pathlib"])
subprocess.check_call(["pip","install","--quiet","--force-reinstall","--no-deps", WHEEL_PATH])
subprocess.check_call(["pip","install","--quiet","dspy>=3.0.4"])
import dspy, fabric_rlm
stage("imported", dspy=dspy.__version__, fabric_rlm=fabric_rlm.__version__)


In [ ]:
rows = []
for line in Path(DATASET_PATH).read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if line: rows.append(json.loads(line))
if SMOKE_N: rows = rows[:SMOKE_N]
stage("dataset_loaded", n=len(rows))

CS_JSON_OBJECT_TEMPLATES = {"HM","MFMC","Scheduling","TM","MCM","LLVM"}
CS_INTEGER_TEMPLATES = {"VLIW","CodeTrace"}
CS_INTEGER_LIST_TEMPLATES = {"Backprop","DistMem"}
INT_RE = re.compile(r"-?\d+")
INT_CSV_RE = re.compile(r"-?\d+(?:\s*,\s*-?\d+)+")

def _resp_text(resp):
    if resp is None: return ""
    return resp if isinstance(resp, str) else str(resp)

def _extract_solution(text):
    if "</think>" in text: text = text.split("</think>", 1)[-1]
    return text.strip() or None

def _extract_last_json_object(text):
    if not text: return None
    last = None; depth = 0; start = -1
    for i, ch in enumerate(text):
        if ch == "{":
            if depth == 0: start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start >= 0:
                last = text[start:i+1]; start = -1
    if not last: return None
    try: return json.loads(last)
    except Exception:
        try: return json.loads(last.replace("'", '"'))
        except Exception: return None

def _parse_int_list(text):
    if not text: return None
    m = INT_CSV_RE.search(text)
    if m: return [int(x.strip()) for x in m.group(0).split(",")]
    nums = INT_RE.findall(text)
    return [int(n) for n in nums] if nums else None

def grade(template, gold_answer, response_text):
    text = _resp_text(response_text); sol = _extract_solution(text) or text
    expected = gold_answer
    if isinstance(expected, str):
        try: expected = json.loads(expected)
        except Exception: pass
    if template in CS_JSON_OBJECT_TEMPLATES:
        cand = _extract_last_json_object(sol) or _extract_last_json_object(text)
        return cand == expected
    if template in CS_INTEGER_TEMPLATES:
        m = INT_RE.search(sol) or INT_RE.search(text)
        if m is None: return False
        try: return int(m.group(0)) == int(str(expected).strip())
        except Exception: return False
    if template in CS_INTEGER_LIST_TEMPLATES:
        if isinstance(expected, list): exp_list = [int(x) for x in expected]
        else: exp_list = _parse_int_list(str(expected))
        pred = _parse_int_list(sol) or _parse_int_list(text)
        return pred == exp_list
    return False

stage("validator_ready")


In [ ]:
# v1 prompt body (frozen historical artifact, restored for arm A only).
# This is the exact pre-commit-0b45c09 build_reflection_prompt body.
from fabric_rlm import prompts as _prompts_mod
_v2_build = _prompts_mod.build_reflection_prompt  # the v2 we ship in the wheel
_format_history = _prompts_mod._format_verifier_history

def _v1_build_reflection_prompt(submitted_payload, original_question=None, verifier_repair_history=None):
    payload_text = repr(submitted_payload)
    if len(payload_text) > 4000: payload_text = payload_text[:3997] + "..."
    question_block = f"Original task:\n{original_question}\n\n" if original_question else ""
    history_block = _format_history(verifier_repair_history)
    return (
        f"{history_block}"
        "You are about to submit the following answer:\n"
        f"<payload>\n{payload_text}\n</payload>\n\n"
        f"{question_block}"
        "Before this is finalized, ATTACK your own answer:\n"
        "1. List the invariants the answer must satisfy (ranges, signs, cross-field consistency, format).\n"
        "2. Confirm the answer is a CONCRETE answer to the task — not a clarification request, "
        "acknowledgement, or 'please confirm' message. If the payload starts with 'Acknowledged', "
        "'Please confirm/clarify/specify/provide', 'I need more information', 'Could you...', "
        "or 'Before I can answer...', it is INVALID — re-SUBMIT with a concrete attempt instead.\n"
        "3. Confirm the answer's shape matches the task. If the prompt enumerates N sub-questions "
        "(Q1..Qn, Part 1..N, numbered list), the SUBMIT must contain exactly N items in the right order. "
        "Short or partial answers are INVALID — re-SUBMIT the full set.\n"
        "4. Write a short Python snippet that asserts each invariant against the submitted values. "
        "If any assertion fails, raise.\n"
        "5. If you find ANY issue, write corrected code that ends with a new SUBMIT(...) call with the fixed payload.\n"
        "6. If the answer survives all checks, print \"REFLECTION_OK: <one-line justification>\" "
        "and do NOT call SUBMIT again.\n"
        "\nThis is your one reflection opportunity for this submission - no further reflection will run."
    )

def use_v1_prompt():
    _prompts_mod.build_reflection_prompt = _v1_build_reflection_prompt
    import fabric_rlm.runtime as _rt
    _rt.build_reflection_prompt = _v1_build_reflection_prompt

def use_v2_prompt():
    _prompts_mod.build_reflection_prompt = _v2_build
    import fabric_rlm.runtime as _rt
    _rt.build_reflection_prompt = _v2_build

stage("prompt_swapper_ready")


In [ ]:
from fabric_rlm import RLM, FabricLM
os.environ["FABRIC_RLM_CAPTURE_TURNS"] = "1"

def run_arm(arm_label, *, effort, enable_reflection, prompt_variant):
    """Run a single arm. prompt_variant in {'v1','v2','none'}."""
    if prompt_variant == "v1": use_v1_prompt()
    elif prompt_variant == "v2": use_v2_prompt()
    arm_dir = RUN_ROOT / arm_label
    arm_dir.mkdir(parents=True, exist_ok=True)
    traces_dir = arm_dir / "traces"; traces_dir.mkdir(exist_ok=True)
    results_path = arm_dir / "results.jsonl"
    summary_path = arm_dir / "summary.json"
    arm_summary = {"arm": arm_label, "effort": effort, "enable_reflection": enable_reflection,
                   "prompt_variant": prompt_variant, "started_at": time.time(), "n": len(rows)}
    lm = FabricLM("gpt-5", reasoning_effort=effort, cache=False)
    stage("arm_start", arm=arm_label, effort=effort, refl=enable_reflection, prompt=prompt_variant)
    n_passed = 0
    with results_path.open("w", encoding="utf-8") as out_fh:
        for idx, row in enumerate(rows):
            qid = row["question_id"]; tpl = row["template"]; gold = row.get("answer")
            rec = {"arm": arm_label, "question_id": qid, "template": tpl, "started_at": time.time()}
            try:
                rlm = RLM(signature="question -> answer", lm=lm, enable_reflection=enable_reflection)
                t0 = time.perf_counter()
                result = rlm.run({"question": row["prompt"]})
                elapsed = time.perf_counter() - t0
                ans = (result.payload or {}).get("answer") if result.payload else None
                passed = bool(result.submitted) and (ans is not None and grade(tpl, gold, ans))
                traj = result.trajectory
                turns = traj.turns if traj is not None else []
                # Identify pre-reflection vs post-reflection submits for harmful/beneficial accounting.
                pre_refl_submit = None; post_refl_submit = None
                for i, t in enumerate(turns):
                    if getattr(t, "submitted", False) and getattr(t, "turn_type", None) != "reflection":
                        pre_refl_submit = t
                    if getattr(t, "turn_type", None) == "reflection" and getattr(t, "submitted", False):
                        post_refl_submit = t
                pre_passed = None; post_passed = None
                if pre_refl_submit is not None and getattr(pre_refl_submit, "submit_payload", None):
                    pre_ans = pre_refl_submit.submit_payload.get("answer")
                    if pre_ans is not None: pre_passed = grade(tpl, gold, pre_ans)
                if post_refl_submit is not None and getattr(post_refl_submit, "submit_payload", None):
                    post_ans = post_refl_submit.submit_payload.get("answer")
                    if post_ans is not None: post_passed = grade(tpl, gold, post_ans)
                # Token totals from turn metadata (v2 wheel records reflect_* fields).
                def _tok(t, k):
                    md = getattr(t, "metadata", None) or {}
                    return md.get(k) or 0
                total_prompt = sum(_tok(t, "prompt_tokens") for t in turns)
                total_completion = sum(_tok(t, "completion_tokens") for t in turns)
                refl_prompt = sum(_tok(t, "prompt_tokens") for t in turns if getattr(t, "turn_type", None) == "reflection")
                refl_completion = sum(_tok(t, "completion_tokens") for t in turns if getattr(t, "turn_type", None) == "reflection")
                rec.update({
                    "passed": bool(passed), "submitted": result.submitted,
                    "elapsed_seconds": elapsed, "n_turns": len(turns),
                    "prompt_tokens": total_prompt, "completion_tokens": total_completion,
                    "reflection_prompt_tokens": refl_prompt,
                    "reflection_completion_tokens": refl_completion,
                    "reflection_revised": post_refl_submit is not None,
                    "pre_reflection_passed": pre_passed,
                    "post_reflection_passed": post_passed,
                    "answer_preview": (str(ans)[:1000] if ans is not None else None),
                })
                if passed: n_passed += 1
                trace = {"arm": arm_label, "question_id": qid, "template": tpl,
                         "prompt": row["prompt"], "answer": str(ans) if ans is not None else None,
                         "submitted": result.submitted, "passed": rec["passed"],
                         "pre_reflection_passed": pre_passed,
                         "post_reflection_passed": post_passed,
                         "reflection_revised": rec["reflection_revised"],
                         "turns": [
                             {"turn_type": getattr(t, "turn_type", None),
                              "submitted": getattr(t, "submitted", False),
                              "code": (getattr(t, "code", None) or "")[:2000],
                              "output": (getattr(t, "output", None) or "")[:2000],
                              "payload_preview": (getattr(t, "submit_payload", None) or {})}
                             for t in turns]}
                (traces_dir / f"trace_{qid}.json").write_text(json.dumps(trace, default=str, indent=2), encoding="utf-8")
            except Exception as exc:
                rec.update({"passed": False, "error": repr(exc), "traceback": traceback.format_exc()})
            out_fh.write(json.dumps(rec, default=str) + "\n"); out_fh.flush()
            stage("q_done", arm=arm_label, idx=idx+1, qid=qid, passed=rec.get("passed"),
                  elapsed=round(rec.get("elapsed_seconds") or 0, 1))
    arm_summary["n_passed"] = n_passed
    arm_summary["pass_rate"] = n_passed / len(rows) if rows else 0
    arm_summary["elapsed"] = time.time() - arm_summary["started_at"]
    summary_path.write_text(json.dumps(arm_summary, indent=2, default=str))
    stage("arm_done", arm=arm_label, n_passed=n_passed, pass_rate=arm_summary["pass_rate"])
    return arm_summary


## Arm B — medium / v2 prompt / reflection ON

In [ ]:
arm_B = run_arm("B_medium_v2", effort="medium", enable_reflection=True, prompt_variant="v2")
print(json.dumps(arm_B, indent=2, default=str))
